This is called a Next Best Offer (NBO) or Recommendation Engine, and it is used by banks worldwide for cross-selling and upselling.

Important: We are not using another ML model here. We are using the lead probability from our ML model together with business rules because banks need recommendations that are explainable.

In [1]:
import pandas as pd
import numpy as np

In [42]:
segment = pd.read_csv("customers_segmented.csv")

segment.head()

,CustomerID,Age,Gender,Occupation,MonthlyIncome,Region,Branch,FCYAccount,ETBAccount,MobileBanking,...,HighRemittance,ActiveCustomer,DigitalCustomer,CustomerValueScore,MonthlyRemittanceAverage,FCYUtilizationRate,RelationshipScore,PotentialFCYCustomer,Cluster,CustomerSegment
0,100001,59,0,0,71321,0,0,1,1,1,...,1,0,0,24368.8,428.75,0.421186,4,0,1,Low Activity Customers
1,100002,49,1,5,144028,1,3,0,1,0,...,1,0,0,48416.4,701.33,0.211383,6,1,0,High Value Customers
2,100003,35,0,9,86131,0,2,0,1,1,...,1,0,0,29201.8,493.75,0.365063,4,0,1,Low Activity Customers
3,100004,63,0,5,37963,7,5,1,1,0,...,0,0,0,13896.4,317.92,0.411271,5,0,2,Frequent Remittance Customers
4,100005,28,1,7,113757,1,3,0,1,0,...,0,0,0,36243.6,252.75,0.532476,4,0,0,High Value Customers


In [43]:
prediction = pd.read_csv("Top_Lead_Customers.csv")

prediction.head()

,CustomerID,Age,Gender,Occupation,MonthlyIncome,Region,Branch,FCYAccount,ETBAccount,MobileBanking,...,FrequentRemittance,HighRemittance,ActiveCustomer,DigitalCustomer,CustomerValueScore,MonthlyRemittanceAverage,FCYUtilizationRate,RelationshipScore,PotentialFCYCustomer,Probability
0,100325,23,0,7,84240,5,7,0,1,1,...,1,1,0,1,28403.5,455.25,0.305327,5,1,0.992390
1,100353,37,0,8,136982,2,5,0,1,1,...,1,1,0,1,45106.6,602.00,0.262874,5,1,0.992239
2,100190,26,0,5,89576,3,6,0,1,1,...,1,0,0,1,29180.8,318.00,0.352463,5,1,0.992239
3,100150,52,1,4,115409,6,0,0,1,1,...,1,1,0,1,39710.7,681.33,0.273239,8,1,0.992141
4,100368,55,1,4,133237,6,2,0,1,1,...,1,0,0,1,42831.6,376.75,0.231365,6,1,0.992141


In [45]:
customers = prediction.merge(
    segment[
        [
            "CustomerID",
            "Cluster",
            "CustomerSegment"
        ]
    ],
    on="CustomerID",
    how="left"
)

In [46]:
customers[["CustomerID","CustomerSegment"]].head()

,CustomerID,CustomerSegment
0,100325,Low Activity Customers
1,100353,High Value Customers
2,100190,Low Activity Customers
3,100150,High Value Customers
4,100368,High Value Customers


In [47]:
customers.columns

Index(['CustomerID', 'Age', 'Gender', 'Occupation', 'MonthlyIncome', 'Region',
       'Branch', 'FCYAccount', 'ETBAccount', 'MobileBanking',
       'InternetBanking', 'RemittanceCount', 'TotalRemittanceUSD',
       'AverageTransactionUSD', 'FCYPurchaseUSD', 'LastTransactionDays',
       'ExistingProducts', 'LeadStatus', 'AgeGroup', 'HighIncome',
       'FrequentRemittance', 'HighRemittance', 'ActiveCustomer',
       'DigitalCustomer', 'CustomerValueScore', 'MonthlyRemittanceAverage',
       'FCYUtilizationRate', 'RelationshipScore', 'PotentialFCYCustomer',
       'Probability', 'Cluster', 'CustomerSegment'],
      dtype='object')

In [48]:
#Create Lead Priority

def lead_priority(prob):

    if prob >= 0.85:
        return "High"

    elif prob >= 0.60:
        return "Medium"

    else:
        return "Low"

In [49]:
customers["LeadPriority"] = customers["Probability"].apply(
    lead_priority
)

In [50]:
print(customers.columns)

Index(['CustomerID', 'Age', 'Gender', 'Occupation', 'MonthlyIncome', 'Region',
       'Branch', 'FCYAccount', 'ETBAccount', 'MobileBanking',
       'InternetBanking', 'RemittanceCount', 'TotalRemittanceUSD',
       'AverageTransactionUSD', 'FCYPurchaseUSD', 'LastTransactionDays',
       'ExistingProducts', 'LeadStatus', 'AgeGroup', 'HighIncome',
       'FrequentRemittance', 'HighRemittance', 'ActiveCustomer',
       'DigitalCustomer', 'CustomerValueScore', 'MonthlyRemittanceAverage',
       'FCYUtilizationRate', 'RelationshipScore', 'PotentialFCYCustomer',
       'Probability', 'Cluster', 'CustomerSegment', 'LeadPriority'],
      dtype='object')


In [51]:
#Verify Priority Distribution

customers["LeadPriority"].value_counts()

LeadPriority
High    263
Low     237
Name: count, dtype: int64

In [10]:
#Create Recommendation Function
def recommend_product(row):

    probability = row["Probability"]

    fcy = row["FCYAccount"]

    remittance = row["TotalRemittanceUSD"]

    income = row["MonthlyIncome"]

    products = row["ExistingProducts"]

    mobile = row["MobileBanking"]

    relationship = row["RelationshipScore"]

    #Rule 1
#High Probability
#No FCY Account

    if probability >= 0.85 and fcy == "No":

        return "Offer FCY Savings Account"

#Rule 2
#Very High Remittance    

    elif remittance >= 8000:

        return "Offer FCY Fixed Deposit"

#Rule 3
#High Income
    elif income >= 120000:

        return "Offer Premium Banking"

#Rule 4
#Strong Relationship

    elif relationship >= 5:

        return "Offer Wealth Management"

#Rule 5
#No Mobile Banking       
    elif mobile == "No":

        return "Promote Mobile Banking"

    else:

        return "General FCY Awareness Campaign"

In [52]:
def recommend_product(row):

    probability = row["Probability"]

    fcy = row["FCYAccount"]

    remittance = row["TotalRemittanceUSD"]

    income = row["MonthlyIncome"]

    relationship = row["RelationshipScore"]

    mobile = row["MobileBanking"]

    if probability >= 0.85 and fcy == "No":

        return "Offer FCY Savings Account"

    elif remittance >= 8000:

        return "Offer FCY Fixed Deposit"

    elif income >= 120000:

        return "Offer Premium Banking"

    elif relationship >= 5:

        return "Offer Wealth Management"

    elif mobile == "No":

        return "Promote Mobile Banking"

    else:

        return "General FCY Awareness Campaign"

In [53]:
#Apply Recommendations

customers["Recommendation"] = customers.apply(

recommend_product,

axis=1
)

In [54]:
#View Results

customers[[

"CustomerID",

"Probability",

"LeadPriority",

"Recommendation"

]].head(20)

,CustomerID,Probability,LeadPriority,Recommendation
0,100325,0.992390,High,Offer Wealth Management
1,100353,0.992239,High,Offer Premium Banking
2,100190,0.992239,High,Offer Wealth Management
3,100150,0.992141,High,Offer FCY Fixed Deposit
4,100368,0.992141,High,Offer Premium Banking
5,100082,0.992026,High,Offer FCY Fixed Deposit
6,100209,0.992026,High,Offer FCY Fixed Deposit
7,100300,0.991985,High,Offer Premium Banking
8,100108,0.991765,High,Offer FCY Fixed Deposit
9,100027,0.991765,High,Offer Premium Banking


In [55]:
#Segment-Based Recommendation
def segment_offer(segment):

    if segment == "High Value Customers":

        return "Relationship Manager Visit"

    elif segment == "Frequent Remittance Customers":

        return "FCY Savings Promotion"

    elif segment == "Digital Banking Customers":

        return "Digital FCY Campaign"

    else:

        return "Standard Marketing Campaign"

In [56]:
customers["Campaign"] = customers["CustomerSegment"].apply(
    segment_offer
)

In [58]:
#Final Action Plan

def action(priority):

    if priority == "High":

        return "Contact within 24 Hours"

    elif priority == "Medium":

        return "Contact within 7 Days"

    else:

        return "Monthly Marketing Campaign"

    customers["ActionPlan"] = customers["LeadPriority"].apply(
    action
)

<function __main__.action(priority)>

In [59]:
#Customer Ranking

customers = customers.sort_values(

by="Probability",

ascending=False
)

In [60]:
#Top 20 Customers

top20 = customers.head(20)

top20

,CustomerID,Age,Gender,Occupation,MonthlyIncome,Region,Branch,FCYAccount,ETBAccount,MobileBanking,...,MonthlyRemittanceAverage,FCYUtilizationRate,RelationshipScore,PotentialFCYCustomer,Probability,Cluster,CustomerSegment,LeadPriority,Recommendation,Campaign
0,100325,23,0,7,84240,5,7,0,1,1,...,455.25,0.305327,5,1,0.992390,1,Low Activity Customers,High,Offer Wealth Management,Standard Marketing Campaign
2,100190,26,0,5,89576,3,6,0,1,1,...,318.00,0.352463,5,1,0.992239,1,Low Activity Customers,High,Offer Wealth Management,Standard Marketing Campaign
1,100353,37,0,8,136982,2,5,0,1,1,...,602.00,0.262874,5,1,0.992239,0,High Value Customers,High,Offer Premium Banking,Relationship Manager Visit
4,100368,55,1,4,133237,6,2,0,1,1,...,376.75,0.231365,6,1,0.992141,0,High Value Customers,High,Offer Premium Banking,Relationship Manager Visit
3,100150,52,1,4,115409,6,0,0,1,1,...,681.33,0.273239,8,1,0.992141,0,High Value Customers,High,Offer FCY Fixed Deposit,Relationship Manager Visit
6,100209,23,0,0,90714,2,1,0,1,1,...,771.67,0.307667,4,1,0.992026,1,Low Activity Customers,High,Offer FCY Fixed Deposit,Standard Marketing Campaign
5,100082,60,0,0,161381,2,2,0,1,1,...,682.92,0.271751,5,1,0.992026,0,High Value Customers,High,Offer FCY Fixed Deposit,Relationship Manager Visit
7,100300,60,1,5,175189,0,2,0,1,1,...,304.00,0.295230,4,1,0.991985,0,High Value Customers,High,Offer Premium Banking,Relationship Manager Visit
9,100027,42,1,2,138503,4,4,0,1,1,...,277.50,0.339339,5,1,0.991765,0,High Value Customers,High,Offer Premium Banking,Relationship Manager Visit
8,100108,55,1,3,149710,2,4,0,1,1,...,723.33,0.252304,7,1,0.991765,0,High Value Customers,High,Offer FCY Fixed Deposit,Relationship Manager Visit


In [61]:
#Recommendation Summary

customers["Recommendation"].value_counts()

Recommendation
Offer Wealth Management           170
Offer Premium Banking             149
General FCY Awareness Campaign     98
Offer FCY Fixed Deposit            83
Name: count, dtype: int64

In [62]:
print(customers.columns)

Index(['CustomerID', 'Age', 'Gender', 'Occupation', 'MonthlyIncome', 'Region',
       'Branch', 'FCYAccount', 'ETBAccount', 'MobileBanking',
       'InternetBanking', 'RemittanceCount', 'TotalRemittanceUSD',
       'AverageTransactionUSD', 'FCYPurchaseUSD', 'LastTransactionDays',
       'ExistingProducts', 'LeadStatus', 'AgeGroup', 'HighIncome',
       'FrequentRemittance', 'HighRemittance', 'ActiveCustomer',
       'DigitalCustomer', 'CustomerValueScore', 'MonthlyRemittanceAverage',
       'FCYUtilizationRate', 'RelationshipScore', 'PotentialFCYCustomer',
       'Probability', 'Cluster', 'CustomerSegment', 'LeadPriority',
       'Recommendation', 'Campaign'],
      dtype='object')


In [63]:
#Campaign Summary

customers.groupby(

"Campaign"

)["Probability"].mean()

Campaign
Digital FCY Campaign           0.548253
FCY Savings Promotion          0.493517
Relationship Manager Visit     0.538028
Standard Marketing Campaign    0.464741
Name: Probability, dtype: float64

In [ ]:
#Branch Summary

branch_summary = customers.groupby("Branch").agg({

"CustomerID":"count",

"Probability":"mean"

}).rename(columns={

"CustomerID":"Customers",

"Probability":"AverageLeadProbability"

})

branch_summary

,Customers,AverageLeadProbability
Branch,,
0,63,0.622760
1,57,0.398465
2,67,0.469259
3,71,0.362704
4,66,0.625002
5,56,0.535386
6,54,0.546750
7,66,0.523894


In [65]:
#Regional Summary

region_summary = customers.groupby("Region").agg({

"CustomerID":"count",

"Probability":"mean"

})

region_summary

,CustomerID,Probability
Region,,
0,61,0.592626
1,55,0.475950
2,71,0.546672
3,57,0.488126
4,59,0.571292
5,64,0.561979
6,63,0.404496
7,70,0.433594


In [66]:
#High Priority Customers

high_priority = customers[

customers["LeadPriority"]=="High"

]

high_priority.head()

,CustomerID,Age,Gender,Occupation,MonthlyIncome,Region,Branch,FCYAccount,ETBAccount,MobileBanking,...,MonthlyRemittanceAverage,FCYUtilizationRate,RelationshipScore,PotentialFCYCustomer,Probability,Cluster,CustomerSegment,LeadPriority,Recommendation,Campaign
0,100325,23,0,7,84240,5,7,0,1,1,...,455.25,0.305327,5,1,0.992390,1,Low Activity Customers,High,Offer Wealth Management,Standard Marketing Campaign
2,100190,26,0,5,89576,3,6,0,1,1,...,318.00,0.352463,5,1,0.992239,1,Low Activity Customers,High,Offer Wealth Management,Standard Marketing Campaign
1,100353,37,0,8,136982,2,5,0,1,1,...,602.00,0.262874,5,1,0.992239,0,High Value Customers,High,Offer Premium Banking,Relationship Manager Visit
4,100368,55,1,4,133237,6,2,0,1,1,...,376.75,0.231365,6,1,0.992141,0,High Value Customers,High,Offer Premium Banking,Relationship Manager Visit
3,100150,52,1,4,115409,6,0,0,1,1,...,681.33,0.273239,8,1,0.992141,0,High Value Customers,High,Offer FCY Fixed Deposit,Relationship Manager Visit


In [67]:
#Save Outputs

customers.to_csv(

"Customer_Recommendations.csv",

index=False

)

branch_summary.to_csv(

"Branch_Summary.csv"

)

region_summary.to_csv(

"Region_Summary.csv"

)

print("Recommendation Engine Completed!")

Recommendation Engine Completed!


Final Output
Customer	Probability	Priority	Recommendation	Campaign	Action
100201	0.98	High	Offer FCY Savings Account	FCY Savings Promotion	Contact within 24 Hours
100187	0.96	High	Offer Premium Banking	Relationship Manager Visit	Contact within 24 Hours
100154	0.93	Medium	Offer FCY Fixed Deposit	Digital FCY Campaign	Contact within 7 Days
How a Bank Would Use This

Every morning, the relationship manager could receive a prioritized list like:

RM	Customer	Recommended Product	Action
RM-01	Customer A	FCY Savings	Call Today
RM-02	Customer B	FCY Fixed Deposit	Visit Tomorrow
RM-03	Customer C	Premium Banking	Schedule Meeting

Instead of manually deciding which customers to contact, the system provides a ranked list with a recommended product and follow-up action.

One Important Improvement

There's one issue with our recommendation logic:

if probability >= 0.85:

These thresholds (0.85, 0.60) are hard-coded. In a production banking environment, they should be based on historical campaign performance, for example:

Top 10% of probabilities → High Priority
Next 20% → Medium Priority
Remaining 70% → Low Priority

This percentile-based approach adapts automatically as the model or customer population changes and is commonly used in production scoring systems.